# Lab 9.2 &mdash; Probes That Can Actually Fail

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 3 &middot; Module 9 &mdash; Deployment &amp; AgentOps**

### What you'll do
- Write liveness and readiness so that they answer different questions
- Simulate the kubelet and find how long a wedged replica keeps traffic
- Measure what pointing liveness at a dependency does to three replicas
- Price a readiness check that calls the model

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, and none of them needs a cluster, so your
> score never depends on a live endpoint or on `kubectl` working. Cells marked **Run it for real**
> do call the sandbox model or your namespace; if either is unreachable they print how to fix it
> instead of crashing.

> **Nothing here needs a cluster.** The kubelet's loop is twenty lines, and you can
> run a thirty-second gateway outage through it in a millisecond. That is a better
> way to learn what `failureThreshold` means than waiting for one.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, math, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-9-02")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

# ---- your own namespace --------------------------------------------------
# You deploy into your own namespace, published at your own host. Both are injected into
# the sandbox, so nothing here is hardcoded and nothing here needs them to be set.
#
# Read ONLY from APP_NAMESPACE, never derived from the hostname. A cell below runs
# kubectl against whatever this says, and a namespace guessed from a machine name is
# the wrong thing to point kubectl at.
APP_NS   = os.environ.get("APP_NAMESPACE", "")
APP_HOST = os.environ.get("APP_HOST", "")

print("work dir :", WORK)
print("model    :", LLM_MODEL or "(not configured -- graded cells still work)")
print("namespace:", APP_NS or "(unknown -- graded cells still work)")

## Concept

Kubernetes asks a pod two different questions and most services answer both the same way.

- **Liveness** &mdash; *is this process broken beyond recovery?* A failure here **restarts the
  container**. It must not depend on anything you do not control.
- **Readiness** &mdash; *should this replica receive traffic right now?* A failure here **removes
  the pod from the Service** and nothing else. It may depend on everything.

An agent service makes the distinction sharp, because its main dependency &mdash; the model
gateway &mdash; is remote, shared, and occasionally slow.

## Section 1 &mdash; Two endpoints, two questions

The trap in this section is not conceptual. It is that a probe reads the **status code** and
never looks at the body.

In [ ]:
GATEWAY = {"up": True}          # the model gateway. The tests below toggle it.

def healthz():
    """Liveness. Is the process alive? Checks nothing downstream, on purpose."""
    return 200, {"status": "ok"}


def readyz():
    """Readiness. Is it safe to send this replica a request?"""
    if GATEWAY["up"]:
        return 200, {"ready": True}
    # TODO: a probe reads the STATUS CODE, not the body. Return the code that means
    # "not yet -- take me out of the Service".
    return BLANK, {"ready": False, "why": "gateway unreachable"}


def readyz_that_cannot_fail():
    """The bug this lab exists for. Looks careful. Is decorative."""
    return 200, {"ready": GATEWAY["up"], "why": None if GATEWAY["up"] else "gateway unreachable"}

In [ ]:
# --- Self-check: Section 1
def with_gateway(up, fn):
    """Call fn() with the gateway up or down, then put it back."""
    was = GATEWAY["up"]
    GATEWAY["up"] = up
    try:
        return fn()
    finally:
        GATEWAY["up"] = was

check("readiness passes while the gateway is up",
      lambda: with_gateway(True, readyz)[0] == 200)
check("READINESS FAILS WITH A STATUS CODE when the gateway is down",
      lambda: with_gateway(False, readyz)[0] == 503,
      "503 is what removes the pod from the Service; a body is never read")
check("liveness passes while the gateway is down",
      lambda: with_gateway(False, healthz)[0] == 200,
      "the process is fine -- restarting it would not bring the gateway back")
check("liveness gives the same answer either way",
      lambda: with_gateway(True, healthz) == with_gateway(False, healthz))
check("the decorative version says the right thing in the body",
      lambda: with_gateway(False, readyz_that_cannot_fail)[1]["ready"] is False)
check("...and STILL RETURNS 200, so the probe can never fail",
      lambda: with_gateway(False, readyz_that_cannot_fail)[0] == 200,
      "this is a readiness check that removes the pod from the Service exactly never")
check("the two endpoints disagree during an outage, which is the point",
      lambda: with_gateway(False, healthz)[0] != with_gateway(False, readyz)[0])

## Section 2 &mdash; The kubelet's loop

`periodSeconds`, `failureThreshold` and `initialDelaySeconds` are the whole of it. Writing the
loop once tells you what the numbers cost, in seconds of traffic sent to a replica that cannot
serve it.

In [ ]:
PERIOD            = 5      # periodSeconds
FAILURE_THRESHOLD = 3      # failureThreshold
INITIAL_DELAY     = 30     # initialDelaySeconds -- must exceed real start-up time
COLD_START        = 20     # how long this app takes to be able to answer at all
REPLICAS          = 3
GATEWAY_DOWN      = (30, 60)     # the gateway is unreachable for 30 seconds
HORIZON           = 120


def gateway_up(t: int) -> bool:
    return not (GATEWAY_DOWN[0] <= t < GATEWAY_DOWN[1])


def probe(endpoint: str, t: int, replica: dict) -> int:
    """What `endpoint` returns for this replica at second t."""
    if t - replica["started"] < COLD_START:
        return 503                                   # not listening yet
    if endpoint == "healthz":
        return 200                                   # the process is up; it checks nothing else
    return 200 if gateway_up(t) else 503             # readyz consults the gateway


def act_now(consecutive_failures: int) -> bool:
    """Has this probe failed enough times in a row for the kubelet to act?"""
    # TODO: one bad probe is a blip, not an outage. The kubelet acts only after
    # failureThreshold CONSECUTIVE failures.
    return BLANK

In [ ]:
def simulate(liveness: str, readiness: str, horizon: int = HORIZON) -> dict:
    """Run REPLICAS replicas through the outage under one probe configuration.

    Returns restarts, the seconds with no ready replica, and the seconds spent serving
    traffic from a replica that cannot actually answer.
    """
    reps = [{"started": -INITIAL_DELAY - 10, "live": 0, "ready_f": 0, "ready": True,
             "restarts": 0} for _ in range(REPLICAS)]
    served = {}
    for t in range(horizon):
        for r in reps:
            if t - r["started"] < INITIAL_DELAY:      # initialDelaySeconds: no probing yet
                r["ready"] = False
                continue
            if t % PERIOD:
                continue
            if probe(liveness, t, r) != 200:
                r["live"] += 1
                if act_now(r["live"]):                # liveness failing RESTARTS the container
                    r.update(started=t, live=0, ready_f=0, ready=False,
                             restarts=r["restarts"] + 1)
                    continue
            else:
                r["live"] = 0
            if probe(readiness, t, r) != 200:
                r["ready_f"] += 1
                if act_now(r["ready_f"]):             # readiness failing only DRAINS traffic
                    r["ready"] = False
            else:
                r["ready_f"], r["ready"] = 0, True
        served[t] = sum(1 for r in reps if r["ready"])

    down = [t for t in range(horizon) if served[t] == 0]
    broken = [t for t in range(horizon) if served[t] > 0 and not gateway_up(t)]
    return {"restarts": sum(r["restarts"] for r in reps),
            "blackout_s": len(down),
            "recovered_at": (max(down) + 1) if down else None,
            "serving_while_broken_s": len(broken)}


def good_config() -> dict:
    """The configuration that drains traffic without destroying warm processes."""
    return {
        # TODO: liveness must not depend on anything you do not control. Which of the two
        # endpoints belongs here -- "healthz" or "readyz"?
        "liveness":  BLANK,
        "readiness": "readyz",
    }


def bad_config() -> dict:
    """Both probes pointed at the same endpoint. The most common mistake there is."""
    return {"liveness": "readyz", "readiness": "readyz"}

In [ ]:
# --- Self-check: Section 2
check("one failed probe is not enough to act on",
      lambda: act_now(1) is False)
check("failureThreshold consecutive failures are",
      lambda: act_now(FAILURE_THRESHOLD) is True)
check("a wedged replica keeps traffic for threshold x period seconds",
      lambda: FAILURE_THRESHOLD * PERIOD == 15,
      "15 seconds of requests go to a replica that already cannot serve them")
check("the good config restarts nothing",
      lambda: simulate(**good_config())["restarts"] == 0)
check("THE BAD CONFIG RESTARTS EVERY REPLICA",
      lambda: simulate(**bad_config())["restarts"] == REPLICAS,
      "the gateway blinked, so Kubernetes killed three healthy processes")
check("both configs drain traffic during the outage -- readiness is doing its job in both",
      lambda: simulate(**good_config())["blackout_s"] > 0
              and simulate(**bad_config())["blackout_s"] > 0)
check("but the bad config stays down after the gateway comes back",
      lambda: simulate(**bad_config())["recovered_at"]
              > simulate(**good_config())["recovered_at"])
check("...by the start-up time it threw away",
      lambda: simulate(**bad_config())["recovered_at"]
              - simulate(**good_config())["recovered_at"] == 10)

In [ ]:
# --- Self-check: Section 2 (continued) -- the probe that cannot fail
def simulate_decorative() -> dict:
    """Readiness returns 200 whatever it thinks. Nothing is ever drained."""
    return simulate("healthz", "healthz")

check("the decorative readiness check produces no blackout at all",
      lambda: simulate_decorative()["blackout_s"] == 0,
      "which looks like the best result on this table")
check("...because it sent every request of the outage to a broken replica",
      lambda: simulate_decorative()["serving_while_broken_s"]
              == GATEWAY_DOWN[1] - GATEWAY_DOWN[0])
check("a real readiness check serves far less traffic it cannot answer",
      lambda: simulate(**good_config())["serving_while_broken_s"]
              < simulate_decorative()["serving_while_broken_s"])

def _table():
    rows = (("healthz + readyz (correct)", good_config()),
            ("readyz + readyz (common)",   bad_config()),
            ("readiness that cannot fail", {"liveness": "healthz", "readiness": "healthz"}))
    print(f"  {'config':30} {'restarts':>9} {'blackout':>9} {'recovered':>10} {'served broken':>14}")
    for label, cfg in rows:
        r = simulate(**cfg)
        rec = f"{r['recovered_at']}s" if r["recovered_at"] is not None else "never down"
        print(f"  {label:30} {r['restarts']:>9} {r['blackout_s']:>8}s "
              f"{rec:>10} {str(r['serving_while_broken_s']) + 's':>14}")
    print(f"\n  The gateway was down for {GATEWAY_DOWN[1] - GATEWAY_DOWN[0]}s in every row.")
guard(_table)

### Read it

Three readings, in the order they usually get argued about.

1. **The bad config recovers later than the outage it was reacting to.** Kubernetes restarted
   three healthy processes, and each then had to get back through `initialDelaySeconds` before it
   could serve anything &mdash; after the gateway was already fine. Restarting is not a neutral
   action; it destroys warm state you paid for.
2. **Readiness alone was enough.** In the correct config nothing restarted, traffic drained, and
   the replicas were serving again on the first probe after the gateway returned.
3. **The decorative check wins the blackout column.** Zero seconds without a ready replica, and
   thirty seconds of requests sent to a replica that could not answer any of them. If you only
   watch availability, this configuration looks like the best of the three.

## Section 3 &mdash; What a readiness check costs

Readiness runs on every replica, forever. That makes it the only code in your service whose
cost is set by `periodSeconds` rather than by traffic.

In [ ]:
def probe_load(replicas: int, period_s: float, check_seconds: float) -> dict:
    """What a dependency-checking readiness probe costs per minute, across the fleet."""
    per_replica_per_min = 60 / period_s
    calls = replicas * per_replica_per_min
    return {"calls_per_min": calls,
            "gateway_seconds_per_min": calls * check_seconds,
            "calls_per_day": calls * 60 * 24}

In [ ]:
# --- Self-check: Section 3
CHEAP = probe_load(REPLICAS, PERIOD, 0.001)     # checks a local flag
REAL  = probe_load(REPLICAS, PERIOD, 0.800)     # calls the model for one token

check("a cheap readiness check costs nothing measurable",
      lambda: CHEAP["gateway_seconds_per_min"] < 0.1)
check("the same probe that calls the model does not",
      lambda: REAL["gateway_seconds_per_min"] > 25)
check("and it does it 51,840 times a day at three replicas",
      lambda: REAL["calls_per_day"] == 51840)
check("halving periodSeconds doubles all of it",
      lambda: probe_load(REPLICAS, PERIOD / 2, 0.8)["calls_per_day"]
              == REAL["calls_per_day"] * 2)

def _cost():
    print(f"  cheap check : {CHEAP['gateway_seconds_per_min']:.3f}s of gateway time per minute")
    print(f"  model check : {REAL['gateway_seconds_per_min']:.1f}s per minute, "
          f"{REAL['calls_per_day']:,.0f} calls per day")
    print("  A readiness probe that calls the model is a load generator you did not plan for,")
    print("  pointed at the dependency you are worried about.")
guard(_cost)

### So what should readiness check?

Check the things that are **local and cheap**: is the client constructed, is the model name
resolved, did the config load, is the in-flight count below the limit. Check the gateway
**passively** &mdash; readiness reads a flag that your request path sets when it sees failures,
rather than generating its own traffic.

That pattern also removes the failure mode where a struggling gateway gets an extra 52,000
calls a day from the health checks of the very service that is waiting on it.

## Run it for real

Time a minimal call to the sandbox gateway, then price the probe you would have written.

In [ ]:
if llm_ready():
    def _price_it():
        t0 = time.perf_counter()
        ask("ok")
        latency = time.perf_counter() - t0
        load = probe_load(REPLICAS, PERIOD, latency)
        print(f"  one minimal call        : {latency:.2f}s")
        print(f"  as a readiness probe    : {load['gateway_seconds_per_min']:.1f}s of gateway "
              f"time per minute, {load['calls_per_day']:,.0f} calls/day")
        print(f"  at 30 participants      : {load['calls_per_day'] * 30:,.0f} calls/day "
              f"before anyone asks a question")
    guard(_price_it)

In [ ]:
score()

## Your turn

1. Add a `startupProbe` to the simulation and remove `initialDelaySeconds`. Show the case it
   handles better: an app whose start-up time varies between 5 and 90 seconds.
2. Implement the passive readiness check described above &mdash; a flag set by the request path,
   with a cool-down. Then find its failure mode: what happens when there is no traffic at all?
3. `terminationGracePeriodSeconds` is the other half of draining. Work out what an agent request
   that has been running for 90 seconds should do when the pod is told to stop.